# 第 24 章 · Tile 模型与 kernel DSL：cuTile、Triton、CuTe DSL、TileLang、Helion

本章回答：前面 23 章都在"替每个线程想"，有没有办法只描述"这个 block 处理哪一块数据、做什么运算"，剩下的交给编译器？CUDA 官方的 Tile 模型是什么？Triton、CuTe DSL、TileLang、Helion 这些 DSL 各在哪一层，什么时候选哪个？

配套讲义：仓库里 `cuda-programming/notes/lessons/24-tile-dsl.md`（讲解、动画、练习都在讲义里）。对应原文：PG §1.2.2.3 Tile Programming in CUDA（`materials/text/cuda-programming-guide.txt` 第 1172–1269 行）、§2.4 Writing Tile Kernels（第 4558–6308 行：声明第 4558 行、启动第 4625 行、建 tile 第 4726 行、读写 tile 第 4879 行、矩阵乘第 5588 行、归约 / scan 第 5687 行）；`docs/blog-cuda-tile-cpp.txt`；[02 生态](../02-cpp-ecosystem.md) §2；[04 调研](../04-resources-and-optimization-survey.md) §3.2（DSL 对比表）；cutile-rs tutorial 01–06

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 SIMT 写法 | 每个线程 `if (i < n)` | `simt_vec_add(a, b, 8)` | = a + b |
| 第二步 Tile 写法 | 每个 block 一整块 | `tile_vec_add(a, b, 8)` | = a + b，代码里没有 threadIdx 和 if |
| 第一步 越界由 tile space 处理 | load 补 0、store 截掉 | `tile_load(a, 2, 8)` | [16, 17, 18, 19, 0, 0, 0, 0] |
| 第一步 2 的幂 | 编译期已知 | `is_valid_tile(s)` | 8、16、128 可以，24 不行 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `is_valid_tile`**（来自 `ch24_tile_model.py`）：tile 每一维必须是 2 的幂（而且编译期已知）。

In [ ]:
def is_valid_tile(size: int) -> bool:
    """tile 每一维必须是 2 的幂（而且编译期已知）。"""
    return size > 0 and size & (size - 1) == 0

**函数 `simt_vec_add`**（来自 `ch24_tile_model.py`）：第 1 章的写法：每个线程算 i，自己写 if (i < n)。

In [ ]:
def simt_vec_add(a: np.ndarray, b: np.ndarray, block_dim: int) -> np.ndarray:
    """第 1 章的写法：每个线程算 i，自己写 if (i < n)。"""
    n = len(a)
    c = np.zeros_like(a)
    grid = (n + block_dim - 1) // block_dim
    for bid in range(grid):
        for tid in range(block_dim):
            i = bid * block_dim + tid
            if i < n:
                c[i] = a[i] + b[i]
    return c

**函数 `tile_load`**（来自 `ch24_tile_model.py`）：ct.load(arr, index=(bid,), shape=(TILE,))：取 tile space 里第 index 块；超出数组的部分补 0。

In [ ]:
def tile_load(arr: np.ndarray, index: int, tile: int) -> np.ndarray:
    """ct.load(arr, index=(bid,), shape=(TILE,))：取 tile space 里第 index 块；超出数组的部分补 0。"""
    out = np.zeros(tile, dtype=arr.dtype)
    part = arr[index * tile:(index + 1) * tile]
    out[:len(part)] = part
    return out

**函数 `tile_store`**（来自 `ch24_tile_model.py`）：ct.store：只写回数组里存在的那部分，越界部分丢掉。

In [ ]:
def tile_store(arr: np.ndarray, index: int, t: np.ndarray):
    """ct.store：只写回数组里存在的那部分，越界部分丢掉。"""
    seg = arr[index * t.size:(index + 1) * t.size]
    seg[:] = t[:len(seg)]

**函数 `tile_vec_add`**（来自 `ch24_tile_model.py`）：Tile 写法：没有 threadIdx、没有 if (i < n)；每个 block 处理一整块，block 里几个线程由编译器定。

In [ ]:
def tile_vec_add(a: np.ndarray, b: np.ndarray, tile: int) -> np.ndarray:
    """Tile 写法：没有 threadIdx、没有 if (i < n)；每个 block 处理一整块，block 里几个线程由编译器定。"""
    assert is_valid_tile(tile), "tile 大小必须是 2 的幂"
    c = np.zeros_like(a)
    for bid in range(-(-len(a) // tile)):           # ct.bid(0)
        ta = tile_load(a, bid, tile)
        tb = tile_load(b, bid, tile)
        tile_store(c, bid, ta + tb)                  # ta + tb 产生一个新 tile，ta、tb 本身不变
    return c

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 n = 20：SIMT（block 8 个线程）和 Tile（TILE = 8）")
a = np.arange(20, dtype=np.float32)
b = np.full(20, 100, dtype=np.float32)
print("  SIMT：", simt_vec_add(a, b, 8)[:5], "…")
print("  Tile：", tile_vec_add(a, b, 8)[:5], "…")
print("  两种写法一样，也等于 a + b：", np.array_equal(simt_vec_add(a, b, 8), a + b)
      and np.array_equal(tile_vec_add(a, b, 8), a + b))
print("  最后一块 tile 的内容（越界补 0）：", tile_load(a, 2, 8))

In [ ]:
print("\n第一步 tile 大小必须是 2 的幂")
for s in (8, 16, 24, 128):
    print(f"  {s:3d}: {is_valid_tile(s)}")

## 3. 代码里才看得到的

1. **`-(-n // tile)` 是向上取整的另一种写法**：Python 的 `//` 向下取整，对负数取两次负号就变成向上取整；和第 1 章的 `(n + B - 1) // B` 结果一样。
2. **`ta + tb` 产生新数组**：对应讲义说的"tile 是不可变的值，每次运算产生新 tile"。
3. **Tile 写法里的"block 里几个线程"根本没出现**：这正是 Tile 模型的意思——由编译器决定。

## 4. 自己改一改

- 把 `tile_vec_add` 的 tile 换成 4、16、32，确认结果不变。
- 试试 `tile_vec_add(a, b, 24)`，看 2 的幂检查怎么报错。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> SIMT 替每个线程想，Tile 替每个 block 想；
> tile 不可变、2 的幂、编译期定，越界编译器管；
> 没有 threadIdx、没有 shared、没有同步，线程数编译器定；
> 从低到高：C++ / PTX → CuTe DSL / ThunderKittens → TileLang / cuTile → Triton → Helion；
> 先 Triton 写对，不够快再往下走；概念到哪一层都一样。